# Model author: declare and inspect one simple resonator

This Notebook is for the person who owns the reusable circuit model. It starts at `CircuitPlan`, names every electrical boundary once, teaches response solving and physical-quantity evaluation separately, then uses both Results together. Open this same `.ipynb` in VS Code to execute it or on GitHub to read its static rendered form. At the current `CONVERGING` checkpoint the package is an API-only scaffold, so executing a construction cell intentionally raises `ScaffoldUnavailableError`.

## Choose the operation before the Spec

| Goal | Call | Spec |
|---|---|---|
| Inspect S/Y/Z over a grid | `run.solve(view, spec)` | `DirectSolveSpec` |
| Evaluate one Direct physical quantity | `run.evaluate(view, spec)` | `DiagonalRootSpec` or another typed quantity Spec |
| Materialize the complete Direct operator | `run.evaluate(view, spec)` | `OperatorSpec` |

A solve-spec class requests a response surface. V1 has no separate `SolveSpec` class: construct `DirectSolveSpec` or `HBSolveSpec` directly. A quantity Spec requests its named physical quantity without an unrelated S/Y/Z sweep. `show()` only presents an existing Result and never executes again.

In [ ]:
from scnsim import (
    CircuitPlan, CircuitRun, DiagonalRootSpec, DirectSolveSpec,
    ReportSpec, SParameterTrace, library as sc, units as u,
)

## 1. The Plan is the physical authority

Components come from an exact Library. Pins are assigned once to named nets or the Plan's one reference. The external port binds a net, not a component pin.

In [ ]:
plan = CircuitPlan(id="simple_readout")

coupling_cap = plan.add(
    sc.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
readout = plan.add(
    sc.grounded_parallel_linear_lc_resonator(
        id="readout",
        subsystem_capacitance=110.0 * u.fF,
        inductance=5.8 * u.nH,
    )
)

plan.reference("ground")
plan.net("signal_in", coupling_cap.pin("a"))
plan.net(
    "readout_node",
    coupling_cap.pin("b"),
    readout.pin("signal"),
)
plan.add_port(
    id="signal_in",
    at="signal_in",
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)

## 2. Solve a response surface with `DirectSolveSpec`

`run.solve(view, direct_spec)` asks for the complete selected-view S/Y/Z response over a grid. This Plan has one external port, so its S family is the 1-by-1 S11 response; SCNSim does not guess S21. `SParameterTrace` names a projection of that complete matrix and does not launch another solve. `run.explain(...)` is the preflight inspection point.

In [ ]:
run = CircuitRun(plan=plan, workspace="results/simple_readout")
view = run.original
frequency_grid = tuple(
    value * u.GHz
    for value in (5.5, 5.6, 5.7, 5.8, 5.9, 6.0, 6.1, 6.2, 6.3)
)

direct_spec = DirectSolveSpec(
    frequencies=frequency_grid,
    traces=(
        SParameterTrace(
            id="reflection",
            input_port="signal_in", input_mode=(),
            output_port="signal_in", output_mode=(),
        ),
    ),
)
run.explain(view, direct_spec).show()
direct = run.solve(view, direct_spec)
direct.s.show(magnitude="db")
direct.traces["reflection"].show(magnitude="db")

## 3. Evaluate one physical quantity with `DiagonalRootSpec`

`run.evaluate(view, readout_root)` asks for the anchored complex root of one named dynamic-operator diagonal. It does not calculate the preceding frequency-grid S/Y/Z response. This Spec is appropriate for a local/bare coordinate; a coupled retained block would use `HybridizedPoleSpec`, while the full labeled matrix would use `OperatorSpec`.

In [ ]:
readout_root = DiagonalRootSpec(
    coordinate="readout_node",
    anchor=6.2 * u.GHz,
)
evaluated = run.evaluate(view, readout_root)
evaluated.show()
evaluated.frequency.to(u.GHz)

## 4. Use the response and physical quantity together

The response Result and root Result remain separate exact requests, but downstream Python and reporting can use both. `ReportSpec` neither solves nor searches for a latest result; it records exactly which Results it presents.

In [ ]:
response_grid = direct.frequencies.to(u.GHz)
root_frequency = evaluated.frequency.to(u.GHz)
response_grid, root_frequency

report = run.build_report(
    ReportSpec(inputs=(direct, evaluated))
)
report.show()